In [65]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [66]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [67]:
#preprocess data
data = data.drop(['RowNumber', 'CustomerId','Surname'], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [68]:
# Lable encoder
lable_encoder_gender = LabelEncoder()
data['Gender'] = lable_encoder_gender.fit_transform(data['Gender'])
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [69]:
# one hot encoding
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [70]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [71]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [72]:
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(), columns = onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [73]:
#combine all columns 
data = pd.concat([data.drop('Geography',axis=1),geo_encoder_df],axis = 1)


In [74]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [75]:
# save the encoded files
with open('lable_encoder_gender.pkl', 'wb') as file:
    pickle.dump(lable_encoder_gender,file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [76]:
# split the data 
X = data.drop('Exited',axis=1)
y = data['Exited']

#train split data 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [77]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN implementation

In [78]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [79]:
## Build ANN model 
model = Sequential([
    Dense(64,activation='relu',input_shape = (X_train.shape[1],)),## HL1 connected with input layer
    Dense(32,activation='relu'), ## HL2
    Dense(1,activation='sigmoid') ## op layer
])

/Users/shyam/shyam clean/ML and AI course/Deep Learning Project/DL_ANN_Project/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [80]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.001)
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x169fb9ee0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [82]:
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

In [83]:
## setting up tensorboard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%y%m%d-%h%m%s")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [84]:
## setting up earlystopping 
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=8,restore_best_weights=True)

In [85]:
## training the model 
history = model.fit(
    X_train,y_train,validation_data = (X_test,y_test), epochs = 100,
    callbacks = [tensorflow_callback,early_stopping_callback]

)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step - accuracy: 0.7979 - loss: 0.4545 - val_accuracy: 0.8345 - val_loss: 0.3941
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 464us/step - accuracy: 0.8405 - loss: 0.3866 - val_accuracy: 0.8630 - val_loss: 0.3531
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.8569 - loss: 0.3541 - val_accuracy: 0.8610 - val_loss: 0.3430
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 464us/step - accuracy: 0.8595 - loss: 0.3442 - val_accuracy: 0.8615 - val_loss: 0.3411
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 0.8611 - loss: 0.3370 - val_accuracy: 0.8610 - val_loss: 0.3426
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 450us/step - accuracy: 0.8619 - loss: 0.3343 - val_accuracy: 0.8580 - val_loss: 0.3422
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 464us/step - accuracy: 0.8639 - loss: 0.3318 - val_accuracy: 0.8570 - val_loss: 0.3411
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 460us/step - accuracy: 0.8656 - loss: 0

In [86]:
model.save('model.h5')

In [87]:
## load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [88]:
%tensorboard --logdir logs/fit260817-Aug081786973717/

Reusing TensorBoard on port 6006 (pid 1663), started 0:22:55 ago. (Use '!kill 1663' to kill it.)